### **LLAMA**

In [ ]:
!pip install -q openai pandas

# ============================================================

import pandas as pd
import time
import re
from openai import OpenAI
from google.colab import files

# ============================================================

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

df = pd.read_csv(input_file)

print("File loaded:", input_file)
print("Columns:", df.columns.tolist())
print("Number of questions:", len(df))


if "question" not in df.columns:
    raise ValueError(
        "Column 'question' not found. Available columns: "
        + str(df.columns.tolist())
    )



if "answer" not in df.columns:
    raise ValueError(
        "Accuracy cannot be calculated because an 'answer' column "
        "was not found.\nAvailable columns: "
        + str(df.columns.tolist())
    )

# ============================================================
# Choose "groq" or "huggingface".
#
# IMPORTANT: as of August 2026 Groq moved Llama-3.1-8B-Instant to
# Enterprise-only pricing (Contact Sales) - it no longer appears on
# Groq's Free or Developer plans. If your Groq key is not on an
# Enterprise plan, the Groq call below will fail with an access/model
# error. Hugging Face still serves this model on its normal Inference
# Providers access, so PROVIDER = "huggingface" is the option that
# works for a regular account.
# ============================================================

PROVIDER = "huggingface"   # "groq" or "huggingface"

if PROVIDER == "groq":
    API_KEY = "key"   # your Groq API key
    client = OpenAI(api_key=API_KEY, base_url="https://api.groq.com/openai/v1")
    MODEL_ID = "llama-3.1-8b-instant"
else:
    from google.colab import userdata
    API_KEY = userdata.get("")   # Colab secret: your Hugging Face token
    client = OpenAI(api_key=API_KEY, base_url="https://router.huggingface.co/v1")
    MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# ============================================================

def extract_number(text):

    # Remove commas from numbers such as 1,000
    text = text.replace(",", "")

    # Look for common answer patterns first
    patterns = [
        r"(?:final answer|answer)\s*(?:is|:)?\s*\$?(-?\d+(?:\.\d+)?)",
        r"####\s*\$?(-?\d+(?:\.\d+)?)"
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)

        if matches:
            return matches[-1]

    # If no explicit answer is found,
    # take the last number in the response
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text)

    if numbers:
        return numbers[-1]

    return None

# ============================================================

def ask_llama(question):

    prompt = f"""
Solve the following GSM8K math problem.

Show your reasoning briefly and then provide the final answer.

At the very end, write the answer in this exact format:

FINAL ANSWER: <number>

Question:
{question}
"""

    start_time = time.perf_counter()

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=800,
        temperature=0
    )

    end_time = time.perf_counter()

    response_time = end_time - start_time

    full_response = response.choices[0].message.content

    model_answer = extract_number(full_response)

    return full_response, model_answer, response_time

##

results = []

for i, row in df.iterrows():

    question_no = i + 1
    question = str(row["question"])
    correct_answer = str(row["answer"]).strip()

    print(f"\nProcessing {question_no}/{len(df)}")

    try:

        full_response, model_answer, response_time = ask_llama(question)

        # Normalize answers before comparison
        correct_normalized = correct_answer.replace(",", "").strip()
        model_normalized = str(model_answer).replace(",", "").strip()

        # Compare numerically where possible
        try:
            is_correct = float(model_normalized) == float(correct_normalized)
        except:
            is_correct = model_normalized == correct_normalized

        results.append({
            "Question_No": question_no,
            "Question": question,
            "Correct_Answer": correct_answer,
            "Model": MODEL_ID,
            "Model_Answer": model_answer,
            "Is_Correct": is_correct,
            "Response_Time_Seconds": round(response_time, 3),
            "Full_Response": full_response
        })

        print("Correct Answer :", correct_answer)
        print("Model Answer   :", model_answer)
        print("Correct?       :", is_correct)
        print("Time           :", round(response_time, 3), "seconds")

    except Exception as e:

        print("Error:", str(e)[:200])

        results.append({
            "Question_No": question_no,
            "Question": question,
            "Correct_Answer": correct_answer,
            "Model": MODEL_ID,
            "Model_Answer": None,
            "Is_Correct": False,
            "Response_Time_Seconds": None,
            "Full_Response": f"ERROR: {str(e)[:300]}"
        })

# ============================================================

results_df = pd.DataFrame(results)

output_file = "llama_3.1_8b_gsm8k_results.csv"
results_df.to_csv(output_file, index=False)

accuracy = results_df["Is_Correct"].mean() * 100
avg_time = results_df["Response_Time_Seconds"].mean()

print("\n============================================")
print(f"Saved results to: {output_file}")
print(f"Accuracy: {accuracy:.1f}%")
print(f"Average response time: {avg_time:.2f} seconds")
print("============================================")

files.download(output_file)

Saving gsm8k_style_50_questions_answers.csv to gsm8k_style_50_questions_answers (3).csv
File loaded: gsm8k_style_50_questions_answers (3).csv
Columns: ['question_id', 'question', 'answer']
Number of questions: 50

Processing 1/50
Correct Answer : 32
Model Answer   : 32
Correct?       : True
Time           : 4.369 seconds

Processing 2/50
Correct Answer : 41
Model Answer   : 41
Correct?       : True
Time           : 3.942 seconds

Processing 3/50
Correct Answer : 78
Model Answer   : 78
Correct?       : True
Time           : 4.863 seconds

Processing 4/50
Correct Answer : 8
Model Answer   : 8
Correct?       : True
Time           : 1.974 seconds

Processing 5/50
Correct Answer : 16
Model Answer   : 16
Correct?       : True
Time           : 3.293 seconds

Processing 6/50
Correct Answer : 183
Model Answer   : 183
Correct?       : True
Time           : 5.383 seconds

Processing 7/50
Correct Answer : 30
Model Answer   : 30
Correct?       : True
Time           : 4.864 seconds

Processing 8/50


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
!pip install -q openai pandas

# ============================================================

import pandas as pd
import time
import re
from openai import OpenAI
from google.colab import files

# ============================================================

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

df = pd.read_csv(input_file)

print("File loaded:", input_file)
print("Columns:", df.columns.tolist())
print("Number of questions:", len(df))


if "question" not in df.columns:
    raise ValueError(
        "Column 'question' not found. Available columns: "
        + str(df.columns.tolist())
    )



if "answer" not in df.columns:
    raise ValueError(
        "Accuracy cannot be calculated because an 'answer' column "
        "was not found.\nAvailable columns: "
        + str(df.columns.tolist())
    )

# ============================================================
# DeepSeek-R1 via Hugging Face Inference Providers.
# Colab Secret needed -> HF_TOKEN (must allow "Make calls to Inference
# Providers"). R1 is a thinking model, so it needs more tokens and time
# than an instruct model like Llama.
# ============================================================

from google.colab import userdata
API_KEY = userdata.get("HF_TOKEN")
client = OpenAI(api_key=API_KEY, base_url="https://router.huggingface.co/v1")
MODEL_ID = "deepseek-ai/DeepSeek-R1"   # add ":fastest" to let the router pick the quickest provider

# ============================================================

def extract_number(text):

    # Remove commas from numbers such as 1,000
    text = text.replace(",", "")

    # Look for common answer patterns first
    patterns = [
        r"(?:final answer|answer)\s*(?:is|:)?\s*\$?(-?\d+(?:\.\d+)?)",
        r"####\s*\$?(-?\d+(?:\.\d+)?)"
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)

        if matches:
            return matches[-1]

    # If no explicit answer is found,
    # take the last number in the response
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text)

    if numbers:
        return numbers[-1]

    return None


def strip_thinking(text):
    # Some providers wrap R1's reasoning in <think>...</think> inside the
    # same content string. Only the part after it is the actual answer.
    if not text:
        return ""
    if "</think>" in text:
        return text.split("</think>")[-1].strip()
    if "<think>" in text:   # thinking never closed (got cut off)
        return ""
    return text.strip()

# ============================================================

def ask_deepseek(question):

    prompt = f"""
Solve the following GSM8K math problem.

Show your reasoning briefly and then provide the final answer.

At the very end, write the answer in this exact format:

FINAL ANSWER: <number>

Question:
{question}
"""

    start_time = time.perf_counter()

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=4000,   # must cover R1's thinking as well as the final answer
        temperature=0.6    # DeepSeek's recommended setting for R1
    )

    end_time = time.perf_counter()

    response_time = end_time - start_time

    choice = response.choices[0]
    raw_content = choice.message.content or ""
    # some providers return the thinking separately from the answer
    reasoning = getattr(choice.message, "reasoning_content", None) or getattr(choice.message, "reasoning", None) or ""

    full_response = strip_thinking(raw_content)
    truncated = (choice.finish_reason == "length")

    model_answer = extract_number(full_response)

    return full_response, reasoning, model_answer, response_time, truncated

##

results = []

for i, row in df.iterrows():

    question_no = i + 1
    question = str(row["question"])
    correct_answer = str(row["answer"]).strip()

    print(f"\nProcessing {question_no}/{len(df)}")

    try:

        full_response, reasoning, model_answer, response_time, truncated = ask_deepseek(question)

        # Normalize answers before comparison
        correct_normalized = correct_answer.replace(",", "").strip()
        model_normalized = str(model_answer).replace(",", "").strip()

        # Compare numerically where possible
        try:
            is_correct = float(model_normalized) == float(correct_normalized)
        except:
            is_correct = model_normalized == correct_normalized

        if truncated:
            is_correct = False

        results.append({
            "Question_No": question_no,
            "Question": question,
            "Correct_Answer": correct_answer,
            "Model": MODEL_ID,
            "Model_Answer": model_answer,
            "Is_Correct": is_correct,
            "Response_Time_Seconds": round(response_time, 3),
            "Truncated": truncated,
            "Full_Response": full_response,
            "Reasoning": reasoning
        })

        print("Correct Answer :", correct_answer)
        print("Model Answer   :", model_answer)
        print("Correct?       :", is_correct)
        print("Time           :", round(response_time, 3), "seconds")
        if truncated:
            print("Note: hit max_tokens before finishing - counted as incorrect")

    except Exception as e:

        print("Error:", str(e)[:200])

        results.append({
            "Question_No": question_no,
            "Question": question,
            "Correct_Answer": correct_answer,
            "Model": MODEL_ID,
            "Model_Answer": None,
            "Is_Correct": False,
            "Response_Time_Seconds": None,
            "Truncated": False,
            "Full_Response": f"ERROR: {str(e)[:300]}",
            "Reasoning": ""
        })

# ============================================================

results_df = pd.DataFrame(results)

output_file = "deepseek_r1_gsm8k_results.csv"
results_df.to_csv(output_file, index=False)

accuracy = results_df["Is_Correct"].mean() * 100
avg_time = results_df["Response_Time_Seconds"].mean()

print("\n============================================")
print(f"Saved results to: {output_file}")
print(f"Accuracy: {accuracy:.1f}%")
print(f"Average response time: {avg_time:.2f} seconds")
print("============================================")

files.download(output_file)

Saving gsm8k_style_50_questions_answers.csv to gsm8k_style_50_questions_answers (4).csv
File loaded: gsm8k_style_50_questions_answers (4).csv
Columns: ['question_id', 'question', 'answer']
Number of questions: 50

Processing 1/50
Correct Answer : 32
Model Answer   : None
Correct?       : False
Time           : 21.45 seconds

Processing 2/50
Correct Answer : 41
Model Answer   : 41
Correct?       : True
Time           : 16.164 seconds

Processing 3/50
Correct Answer : 78
Model Answer   : 78
Correct?       : True
Time           : 24.619 seconds

Processing 4/50
Correct Answer : 8
Model Answer   : None
Correct?       : False
Time           : 20.167 seconds

Processing 5/50
Correct Answer : 16
Model Answer   : None
Correct?       : False
Time           : 20.598 seconds

Processing 6/50
Correct Answer : 183
Model Answer   : 183
Correct?       : True
Time           : 15.933 seconds

Processing 7/50
Correct Answer : 30
Model Answer   : None
Correct?       : False
Time           : 30.398 second

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>